In [0]:
# notebook 2 - camada SILVER
# le a bronze, limpa, coloca os nomes em portugues e grava as 7 tabelas silver com o nome que a atividade pede
# a gold le exatamente esses nomes: tb_info_filmes, tb_financeiro_filmes, tb_metricas_engajamento,
# tb_avaliacoes_usuarios, tb_generos, tb_pessoas_empresas e tb_cotacao_dolar
import importlib, sys
if 'medalhao_config' in sys.modules:
    del sys.modules['medalhao_config']
from medalhao_config import *
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
print(bronze_schema, silver_schema)

medalhao_cinema.bronze medalhao_cinema.silver


In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime
from pyspark.sql.window import Window


dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

In [0]:
# funcoes que repito em todas as tabelas
# mais_recente: quando o mesmo id vem 2x na bronze (append), fico so com a linha da ingestao mais nova
# numero_seguro: cast que devolve NULL em vez de quebrar quando vem texto (ANSI do databricks quebra no cast normal)
def mais_recente(df, chave):
    w = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

def numero_seguro(coluna, tipo):
    return F.expr(f"try_cast({coluna} as {tipo})")

def salvar_silver(df, nome):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table(silver_schema, nome))
    print(f"gravado: {table(silver_schema, nome)} ({df.count()} linhas)")

## silver.tb_info_filmes

In [0]:
# pegando como referencia o uso de regex que foi falado na aula pratica, junto da parte que uso como base o notebook 
# consertado: nome da tabela (tb_info_filmes), dedup pela ingestao mais recente, data com try (nao quebra no ANSI)
# e o status normalizado uma vez so (tudo que nao e letra vira espaco) antes de traduzir

df_bronze_filmes=spark.table(table(bronze_schema,"tb_movies_info"))
                             
df_silver_filmes=(df_bronze_filmes
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("title","titulo")
    .withColumnRenamed("original_title","titulo_original")
    .withColumnRenamed("release_date","data_lancamento")
    .withColumnRenamed("runtime","duracao_minutos")
    .withColumnRenamed("original_language","idioma_original")
    .withColumnRenamed("status","status_filme")
    .withColumnRenamed("overview","sinopse")
    .withColumnRenamed("tagline","frase_divulgacao")
)

df_silver_filmes=df_silver_filmes.withColumn("id_filme",F.trim(F.col("id_filme").cast("string")))
df_silver_filmes=mais_recente(df_silver_filmes.filter(F.col("id_filme").isNotNull()),"id_filme")

status_limpo=F.trim(F.regexp_replace(F.upper(F.col("status_filme")),"[^A-Z]+"," "))

df_silver_filmes=df_silver_filmes.withColumn(
    "status_filme",
    F.when(status_limpo=="RELEASED","Lançado")
     .when(status_limpo=="POST PRODUCTION","Pós-Produção")
     .when(status_limpo=="IN PRODUCTION","Em Produção")
     .when(status_limpo=="PLANNED","Planejado")
     .when(status_limpo=="RUMORED","Rumores")
     .when(status_limpo=="CANCELED","Cancelado")
     .otherwise("Não Informado")
)

data_txt=F.trim(F.col("data_lancamento").cast("string"))
df_silver_filmes=df_silver_filmes.withColumn(
    "data_lancamento",
    F.coalesce(
        F.to_date(F.try_to_timestamp(data_txt,F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(data_txt,F.lit("dd/MM/yyyy"))),
        F.to_date(F.try_to_timestamp(data_txt,F.lit("MM/dd/yyyy"))),
        F.to_date(F.try_to_timestamp(data_txt,F.lit("yyyy/MM/dd"))),
        F.to_date(F.try_to_timestamp(data_txt,F.lit("dd-MM-yyyy")))
    )
).withColumn("ano_lancamento",F.year("data_lancamento"))

df_silver_filmes=(df_silver_filmes
    .withColumn("duracao_minutos",numero_seguro("duracao_minutos","double").cast("int"))
    .select("id_filme","titulo","titulo_original","data_lancamento","ano_lancamento","duracao_minutos",
            "idioma_original","status_filme","sinopse","frase_divulgacao","ingestion_datetime"))

dq_check("silver.tb_info_filmes","id_filme não nulo",df_silver_filmes,F.col("id_filme").isNotNull())
dq_check_unique("silver.tb_info_filmes","id_filme único",df_silver_filmes,["id_filme"])

# cria o schema e salva com o nome pedido na atividade
salvar_silver(df_silver_filmes,"tb_info_filmes")
display(df_silver_filmes.limit(10))

[PASS] silver.tb_info_filmes | id_filme não nulo | 0/97879 linhas falharam
[PASS] silver.tb_info_filmes | id_filme único | 0 chaves duplicadas de 97879 linhas
gravado: medalhao_cinema.silver.tb_info_filmes (97879 linhas)


id_filme,titulo,titulo_original,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ingestion_datetime
1000079,Pourquoi tu souris ?,Pourquoi tu souris ?,2024-06-26,2024,0,fr,Em Produção,null,null,2026-10-07T21:44:44.150Z
1000108,Le Petit blond de la casbah,Le Petit blond de la casbah,2023-11-15,2023,0,fr,Pós-Produção,null,null,2026-10-07T21:44:44.150Z
1000219,"Maman, ne me laisse pas m'endormir","Maman, ne me laisse pas m'endormir",2022-10-18,2022,90,fr,Lançado,"Juliette discovers that her teenage son, whom she thought was flourishing, is in reality undermined by a deep malaise and that he has become dangerously dependent on drugs. Determined to get him out of his deadly addiction, and driven by her unconditional love, she embarks on a long and painful fight, alongside her child, without ever giving up on saving him.",null,2026-10-07T21:44:44.150Z
1000251,Butterfly,Butterfly,2019-09-06,2019,23,en,Lançado,A fashion model drives into the desert and meets a mysterious stranger who finally brings the change she's been looking for.,null,2026-10-07T21:44:44.150Z
1000409,Strangers by Night,Une nuit,2023-07-05,2023,90,fr,Lançado,\In a crowded subway train,"a woman pushes a man. They argue. Their little joust """"\""""barks loudly""""\""""""""",2026-10-07T21:44:44.150Z
1000517,The Temple Woods Gang,Le gang des bois du temple,2022-10-16,2022,116,fr,Lançado,"A retired military man lives in the Temple Woods housing project. Just as he’s burying his mother, his neighbour Bébé, who belongs to a gang of robbers from the area, is preparing to rob the convoy of a wealthy Arab prince.",null,2026-10-07T21:44:44.150Z
1000521,Princes of the Desert,"Zodi et Téhu, frères du désert",2023-02-08,2023,105,fr,Lançado,"Zodi, a 12 years old nomad, find an orphan baby dromedary in the desert. He takes in the animal, feeds it, calls it Téhu and becomes its best friend.",null,2026-10-07T21:44:44.150Z
1000553,We Bought A Haunted House,We Bought A Haunted House,2018-01-02,2018,45,en,Lançado,\John and Kathy Burns have purchased the house of their dreams,not realizing that it is haunted by the ghost of a murder victim. . They move into their new home on Halloween,2026-10-07T21:44:44.150Z
1000603,The Small Victories,Les petites victoires,2023-03-01,2023,89,fr,Lançado,"Between her roles as mayor and teacher in the small village of Kerguen, Alice's days are very full. When an unexpected new student, 60-year-old Emile, finally decides to learn to read and write, her daily life threatens to become unmanageable. Especially since Alice will also have to save her village and her school...",null,2026-10-07T21:44:44.150Z
1001005,Twin Peaks: The Phenomenon,Twin Peaks: The Phenomenon,2017-12-05,2017,14,en,Lançado,"\""""\""""Twin Peaks: The Phenomenon""""\"""" is a three-part short documentary briefly chronicling the history of Twin Peaks. Produced and released on YouTube as part of the build-up to the premiere of the 2017 series""""",null,2026-10-07T21:44:44.150Z


## silver.tb_cotacao_dolar

In [0]:
# faço a cotacao antes do financeiro porque o financeiro precisa dela pra calcular o BRL
# a API nao tem fim de semana/feriado: crio todos os dias do periodo e preencho o buraco com o ultimo dia util (forward fill)
df_bronze_cotacao=spark.table(table(bronze_schema,"tb_cotacao_dolar"))

w_dia=Window.partitionBy("data_cotacao").orderBy(F.col("data_hora_cotacao").desc(),F.col("ingestion_datetime").desc())
cotacao_dia=(df_bronze_cotacao
    .withColumn("data_cotacao",F.to_date("data_hora_cotacao"))
    .withColumn("rn",F.row_number().over(w_dia))
    .filter("rn = 1")
    .select("data_cotacao","cotacao_compra","cotacao_venda"))

limites=cotacao_dia.agg(F.min("data_cotacao").alias("ini"),F.max("data_cotacao").alias("fim"))
calendario=limites.select(F.explode(F.sequence("ini",F.greatest("fim",F.current_date()))).alias("data_cotacao"))

w_ffill=Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding,0)
df_silver_cotacao=(calendario
    .join(cotacao_dia,"data_cotacao","left")
    .withColumn("dia_preenchido",F.col("cotacao_compra").isNull())
    .withColumn("cotacao_compra",F.last("cotacao_compra",ignorenulls=True).over(w_ffill).cast("decimal(10,4)"))
    .withColumn("cotacao_venda",F.last("cotacao_venda",ignorenulls=True).over(w_ffill).cast("decimal(10,4)")))

dq_check_unique("silver.tb_cotacao_dolar","1 linha por dia",df_silver_cotacao,["data_cotacao"])
dq_check("silver.tb_cotacao_dolar","cotacao preenchida",df_silver_cotacao,F.col("cotacao_compra").isNotNull())

salvar_silver(df_silver_cotacao,"tb_cotacao_dolar")
display(df_silver_cotacao.orderBy(F.col("data_cotacao").desc()))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


[PASS] silver.tb_cotacao_dolar | 1 linha por dia | 0 chaves duplicadas de 37 linhas
[PASS] silver.tb_cotacao_dolar | cotacao preenchida | 0/37 linhas falharam
gravado: medalhao_cinema.silver.tb_cotacao_dolar (37 linhas)


data_cotacao,cotacao_compra,cotacao_venda,dia_preenchido
2026-10-07,4.9929,4.9935,false
2026-10-06,4.9692,4.9698,false
2026-10-05,4.9853,4.9859,false
2026-10-04,5.2232,5.2238,true
2026-10-03,5.2232,5.2238,true
2026-10-02,5.2232,5.2238,false
2026-10-01,5.2073,5.2079,false
2026-09-30,5.1803,5.1809,false
2026-09-29,5.2198,5.2204,false
2026-09-28,5.2126,5.2132,false


## silver.tb_financeiro_filmes

In [0]:
# limpa orcamento/receita: texto de ausencia vira NULL, tira simbolo de moeda e ponto/virgula de milhar, <= 0 vira NULL
# BRL = USD x cotacao de compra mais recente da silver.tb_cotacao_dolar
# lucro so e calculado quando tem orcamento e receita (senao fica NULL, nao inventa zero); margem so quando receita > 0
def limpar_valor(coluna):
    txt=F.upper(F.trim(F.col(coluna).cast("string")))
    txt=F.when(txt.isin("UNKNOWN","NÃO INFORMADO","NAO INFORMADO","N/A","NA","NULL","NONE",""),None).otherwise(txt)
    txt=F.regexp_replace(txt,"[^0-9.,-]","")
    txt=(F.when(txt.rlike(r"^-?\d{1,3}([.,]\d{3})+$"),F.regexp_replace(txt,"[.,]",""))
          .when(txt.rlike(r"\.\d*,"),F.regexp_replace(F.regexp_replace(txt,"\\.",""),",","."))
          .otherwise(F.regexp_replace(txt,",","")))
    return txt

cotacao=(spark.table(table(silver_schema,"tb_cotacao_dolar"))
    .filter(F.col("data_cotacao")<=F.current_date())
    .orderBy(F.col("data_cotacao").desc())
    .first()["cotacao_compra"])
print("cotacao usada:",cotacao)

df_bronze_fin=spark.table(table(bronze_schema,"tb_movies_financials"))

df_silver_fin=(df_bronze_fin
    .withColumn("id_filme",F.trim(F.col("id").cast("string")))
    .filter(F.col("id_filme").isNotNull())
    .withColumn("orcamento_txt",limpar_valor("budget"))
    .withColumn("receita_txt",limpar_valor("revenue")))
df_silver_fin=mais_recente(df_silver_fin,"id_filme")

df_silver_fin=(df_silver_fin
    .withColumn("orcamento_usd",numero_seguro("orcamento_txt","decimal(18,2)"))
    .withColumn("receita_usd",numero_seguro("receita_txt","decimal(18,2)"))
    .withColumn("orcamento_usd",F.when(F.col("orcamento_usd")>0,F.col("orcamento_usd")))
    .withColumn("receita_usd",F.when(F.col("receita_usd")>0,F.col("receita_usd")))
    .withColumn("cotacao_usada",F.lit(cotacao).cast("decimal(10,4)"))
    .withColumn("orcamento_brl",F.round(F.col("orcamento_usd")*F.col("cotacao_usada"),2).cast("decimal(18,2)"))
    .withColumn("receita_brl",F.round(F.col("receita_usd")*F.col("cotacao_usada"),2).cast("decimal(18,2)"))
    .withColumn("lucro_usd",(F.col("receita_usd")-F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl",(F.col("receita_brl")-F.col("orcamento_brl")).cast("decimal(18,2)"))
    .withColumn("margem_lucro_percentual",
        F.when(F.col("receita_usd")>0,F.round(F.col("lucro_usd")/F.col("receita_usd")*100,2)).cast("decimal(10,2)"))
    .select("id_filme","orcamento_usd","receita_usd","orcamento_brl","receita_brl","lucro_usd","lucro_brl",
            "margem_lucro_percentual","cotacao_usada","ingestion_datetime"))

dq_check_unique("silver.tb_financeiro_filmes","id_filme único",df_silver_fin,["id_filme"])
dq_check("silver.tb_financeiro_filmes","orcamento > 0 ou NULL",df_silver_fin,F.col("orcamento_usd").isNull()|(F.col("orcamento_usd")>0))

salvar_silver(df_silver_fin,"tb_financeiro_filmes")
display(df_silver_fin.limit(10))

cotacao usada: 4.9929
[PASS] silver.tb_financeiro_filmes | id_filme único | 0 chaves duplicadas de 99006 linhas
[PASS] silver.tb_financeiro_filmes | orcamento > 0 ou NULL | 0/99006 linhas falharam
gravado: medalhao_cinema.silver.tb_financeiro_filmes (99006 linhas)


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual,cotacao_usada,ingestion_datetime
1000004,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000005,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000011,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000030,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000073,6000000.00,null,29957400.00,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000081,5.00,null,24.96,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000091,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000092,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000094,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z
1000096,null,null,null,null,null,null,null,4.9929,2026-10-07T21:44:49.481Z


## silver.tb_metricas_engajamento

In [0]:
# popularidade vem com ponto e virgula misturados: acerto o separador decimal antes do cast
# column shift: texto nas colunas de nota/voto vira NULL pelo try_cast (nao quebra o pipeline)
# nota fora de 0-10 (ex.: 78 por erro de escala) vira NULL; voto/popularidade negativo vira NULL
pop=F.regexp_replace(F.trim(F.col("popularity").cast("string")),"[^0-9.,-]","")
pop=(F.when(pop.rlike(r"\.\d*,"),F.regexp_replace(F.regexp_replace(pop,"\\.",""),",","."))   
      .when(pop.rlike(r",\d*\."),F.regexp_replace(pop,",",""))                                 
      .otherwise(F.regexp_replace(pop,",",".")))                                                 

def nota_txt(coluna):
    return F.regexp_replace(F.trim(F.col(coluna).cast("string")),",",".")

df_bronze_met=spark.table(table(bronze_schema,"tb_movies_metrics"))

df_silver_met=(df_bronze_met
    .withColumn("id_filme",F.trim(F.col("id").cast("string")))
    .filter(F.col("id_filme").isNotNull())
    .withColumn("pop_txt",pop)
    .withColumn("tmdb_txt",nota_txt("vote_average"))
    .withColumn("imdb_txt",nota_txt("averageRating"))
    .withColumn("votos_tmdb_txt",nota_txt("vote_count"))
    .withColumn("votos_imdb_txt",nota_txt("numVotes")))
df_silver_met=mais_recente(df_silver_met,"id_filme")

df_silver_met=(df_silver_met
    .withColumn("popularidade",numero_seguro("pop_txt","double"))
    .withColumn("nota_media_tmdb",numero_seguro("tmdb_txt","double"))
    .withColumn("nota_media_imdb",numero_seguro("imdb_txt","double"))
    .withColumn("qtd_votos_tmdb",numero_seguro("try_cast(votos_tmdb_txt as double)","int"))
    .withColumn("qtd_votos_imdb",numero_seguro("try_cast(votos_imdb_txt as double)","int"))
    .withColumn("popularidade",F.when(F.col("popularidade")>=0,F.col("popularidade")))
    .withColumn("nota_media_tmdb",F.when(F.col("nota_media_tmdb").between(0,10),F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb",F.when(F.col("nota_media_imdb").between(0,10),F.col("nota_media_imdb")))
    .withColumn("qtd_votos_tmdb",F.when(F.col("qtd_votos_tmdb")>=0,F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb",F.when(F.col("qtd_votos_imdb")>=0,F.col("qtd_votos_imdb")))
    .select("id_filme","popularidade","nota_media_tmdb","qtd_votos_tmdb","nota_media_imdb","qtd_votos_imdb","ingestion_datetime"))

dq_check_unique("silver.tb_metricas_engajamento","id_filme único",df_silver_met,["id_filme"])
dq_check("silver.tb_metricas_engajamento","nota tmdb 0-10 ou NULL",df_silver_met,
         F.col("nota_media_tmdb").isNull()|F.col("nota_media_tmdb").between(0,10))

salvar_silver(df_silver_met,"tb_metricas_engajamento")
display(df_silver_met.limit(10))

[PASS] silver.tb_metricas_engajamento | id_filme único | 0 chaves duplicadas de 99013 linhas
[PASS] silver.tb_metricas_engajamento | nota tmdb 0-10 ou NULL | 0/99013 linhas falharam
gravado: medalhao_cinema.silver.tb_metricas_engajamento (99013 linhas)


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb,ingestion_datetime
1000004,1.132,0.0,0,6.8,27,2026-10-07T21:44:52.865Z
1000005,0.6,0.0,0,null,40,2026-10-07T21:44:52.865Z
1000030,0.615,0.0,0,null,25,2026-10-07T21:44:52.865Z
1000081,102.802,5.142,134,5.1,1841,2026-10-07T21:44:52.865Z
1000092,4.797,null,null,null,130,2026-10-07T21:44:52.865Z
1000094,2.645,6.9,28,6.5,null,2026-10-07T21:44:52.865Z
1000096,0.71,10.0,1,6.3,695,2026-10-07T21:44:52.865Z
1000108,3.231,0.0,0,6.5,157,2026-10-07T21:44:52.865Z
1000194,6.191,6.554,57,7.0,1652,2026-10-07T21:44:52.865Z
1000197,2.552,6.0,4,5.9,183,2026-10-07T21:44:52.865Z


## silver.tb_avaliacoes_usuarios

In [0]:
# nota fora de 0-10 vira NULL; comentario vazio ou so espaco vira "Sem comentário"
# depois de limpar tiro as avaliacoes 100% iguais (filme + usuario + nota + comentario)
df_bronze_rev=spark.table(table(bronze_schema,"tb_movies_reviews"))

df_silver_rev=(df_bronze_rev
    .withColumn("id_filme",F.trim(F.col("id").cast("string")))
    .withColumn("nome_usuario",F.trim(F.col("nome").cast("string")))
    .withColumn("nota_txt",F.regexp_replace(F.trim(F.col("nota").cast("string")),",","."))
    .withColumn("nota_usuario",numero_seguro("nota_txt","double"))
    .withColumn("nota_usuario",F.when(F.col("nota_usuario").between(0,10),F.col("nota_usuario")))
    .withColumn("comentario_usuario",
        F.when(F.col("comentario").isNull()|(F.trim(F.col("comentario").cast("string"))==""),"Sem comentário")
         .otherwise(F.trim(F.col("comentario").cast("string"))))
    .filter(F.col("id_filme").isNotNull())
    .select("id_filme","nome_usuario","nota_usuario","comentario_usuario")
    .dropDuplicates(["id_filme","nome_usuario","nota_usuario","comentario_usuario"]))

dq_check("silver.tb_avaliacoes_usuarios","nota 0-10 ou NULL",df_silver_rev,
         F.col("nota_usuario").isNull()|F.col("nota_usuario").between(0,10))
dq_check_unique("silver.tb_avaliacoes_usuarios","avaliacao única",df_silver_rev,
                ["id_filme","nome_usuario","nota_usuario","comentario_usuario"])

salvar_silver(df_silver_rev,"tb_avaliacoes_usuarios")
display(df_silver_rev.limit(10))

[PASS] silver.tb_avaliacoes_usuarios | nota 0-10 ou NULL | 0/32412 linhas falharam
[PASS] silver.tb_avaliacoes_usuarios | avaliacao única | 0 chaves duplicadas de 32412 linhas
gravado: medalhao_cinema.silver.tb_avaliacoes_usuarios (32412 linhas)


id_filme,nome_usuario,nota_usuario,comentario_usuario
637007,Lucas Reis 602,3.9,Sem comentário
1100094,Gabriel Carvalho 581,6.2,"Aceitável, mas esperava mais."
628575,Alexandre Barbosa 220,0.3,Péssimo em todos os sentidos.
573249,Rodrigo Oliveira 273,0.5,Péssimo em todos os sentidos.
592539,Pedro Costa 181,4.8,"Não gostei, história confusa."
464493,Adriana Dias 257,0.4,Péssimo em todos os sentidos.
1199748,Eduardo Dias 177,6.0,"Poderia ser melhor, mas não é ruim."
640543,Cristina Monteiro 310,6.4,Sem comentário
599134,Larissa Lopes 330,2.6,Péssimo em todos os sentidos.
424011,Vinícius Ferreira 405,0.5,Não recomendo de jeito nenhum.


## silver.tb_generos

In [0]:
# troco ; por , e faco split + explode: 1 linha por filme x genero
# limpo o lixo do column shift: vazio, numero e texto descritivo (genero tem no maximo 3 palavras e so letras)
df_bronze_cred=mais_recente(
    spark.table(table(bronze_schema,"tb_credits_and_tags")).withColumn("id_filme",F.trim(F.col("id").cast("string"))),
    "id_filme")

df_silver_generos=(df_bronze_cred
    .filter(F.col("id_filme").isNotNull())
    .withColumn("nome_genero",F.explode(F.split(F.regexp_replace(F.col("genres").cast("string"),";",","),",")))
    .withColumn("nome_genero",F.initcap(F.trim(F.col("nome_genero"))))
    .filter(F.col("nome_genero").rlike("^[A-Za-zÀ-ÿ&' -]{2,30}$"))
    .filter(F.size(F.split(F.col("nome_genero")," "))<=3)
    .select("id_filme","nome_genero")
    .dropDuplicates(["id_filme","nome_genero"]))

dq_check_unique("silver.tb_generos","filme x genero único",df_silver_generos,["id_filme","nome_genero"])

salvar_silver(df_silver_generos,"tb_generos")
display(df_silver_generos.groupBy("nome_genero").count().orderBy(F.col("count").desc()))

[PASS] silver.tb_generos | filme x genero único | 0 chaves duplicadas de 132718 linhas
gravado: medalhao_cinema.silver.tb_generos (132718 linhas)


nome_genero,count
Drama,30760
Documentary,18847
Comedy,17518
Thriller,9519
Horror,9257
Romance,7047
Action,5570
Crime,4349
Animation,4197
Tv Movie,3703


## silver.tb_pessoas_empresas

In [0]:
# junto as 4 colunas numa tabela so: cast=Ator, directors=Diretor, writers=Roteirista, production_companies=Produtora
# split por , ou ; + explode, capitalizacao padrao (initcap), tiro vazio/numero/texto longo e duplicados
def explodir(coluna,tipo):
    return (df_bronze_cred
        .select("id_filme",F.explode(F.split(F.col(coluna).cast("string"),"[,;]")).alias("nome_entidade"))
        .withColumn("nome_entidade",F.initcap(F.trim(F.regexp_replace(F.col("nome_entidade"),"\\s+"," "))))
        .withColumn("tipo_entidade",F.lit(tipo)))

df_silver_pessoas=(explodir("cast","Ator")
    .unionByName(explodir("directors","Diretor"))
    .unionByName(explodir("writers","Roteirista"))
    .unionByName(explodir("production_companies","Produtora"))
    .filter(F.col("id_filme").isNotNull())
    .filter(F.col("nome_entidade").isNotNull()&(F.col("nome_entidade")!=""))
    .filter(~F.col("nome_entidade").rlike("^[0-9.,\\s-]+$"))
    .filter((F.length("nome_entidade")<=60)&(F.size(F.split(F.col("nome_entidade")," "))<=6))
    .select("id_filme","nome_entidade","tipo_entidade")
    .dropDuplicates(["id_filme","nome_entidade","tipo_entidade"]))

dq_check("silver.tb_pessoas_empresas","tipo válido",df_silver_pessoas,
         F.col("tipo_entidade").isin("Ator","Diretor","Roteirista","Produtora"))
dq_check_unique("silver.tb_pessoas_empresas","sem duplicado",df_silver_pessoas,["id_filme","nome_entidade","tipo_entidade"])

salvar_silver(df_silver_pessoas,"tb_pessoas_empresas")
display(df_silver_pessoas.groupBy("tipo_entidade").count())

[PASS] silver.tb_pessoas_empresas | tipo válido | 0/906012 linhas falharam
[PASS] silver.tb_pessoas_empresas | sem duplicado | 0 chaves duplicadas de 906012 linhas
gravado: medalhao_cinema.silver.tb_pessoas_empresas (906012 linhas)


tipo_entidade,count
Roteirista,137167
Produtora,119119
Diretor,107801
Ator,541925


In [0]:
# conferencia: tem que aparecer as 7 tabelas aqui antes de rodar a gold
display(spark.sql(f"SHOW TABLES IN {silver_schema}"))

database,tableName,isTemporary
silver,filmes,false
silver,tb_avaliacoes_usuarios,false
silver,tb_cotacao_dolar,false
silver,tb_financeiro_filmes,false
silver,tb_generos,false
silver,tb_info_filmes,false
silver,tb_metricas_engajamento,false
silver,tb_pessoas_empresas,false
